# SafeQuery on Colab (Run 5) - restore from Drive, re-evaluate with fixes A-C
**Lexsi Labs hackathon - Agentic Fine-tuning track** - AgentTune (training) + AuditKIT (evaluation) on **Tiny Aya Global**.

Run 3 trained the SFT adapter and saved everything to `MyDrive/safequery`. **Run 4 does not retrain.** It:
1. restores the saved files from Drive (SFT adapter, results, task files) and checks them,
2. rebuilds the database (deterministic, seed 7) and checks the rebuilt tasks match the saved ones,
3. keeps the 60-task Run 3 results as `results_run3_60/`, then re-runs base + SFT on **all 111 test tasks**,
4. re-runs BFCL + ARC with a **cleared AuditKIT cache** (Run 3's SFT benchmark numbers were cached copies of base),
5. compares, writes the ROI, saves to Drive and downloads.

**Runtime:** A100 (recommended) or L4. A T4 also works but is ~3x slower. Native bf16 is picked automatically.

| Step | What | A100 / L4 (est.) |
|---|---|---|
| 0-1 | code, install, login | ~10 min |
| R | restore from Drive + checks | ~2 min |
| 2 | rebuild db + smoke test | ~3 min |
| 3 | base eval, 111 tasks | ~10-20 min |
| 5 | SFT eval, 111 tasks | ~15-40 min |
| 6 | BFCL + ARC, both models | ~20-40 min |
| 7 | *optional* GRPO | off by default |
| 8-10 | compare, ROI, download, push | ~5 min |

**Fixes in this zip (upload the `safequery_agent.zip` rebuilt on 2026-11-08 for Run 5):**
- **A** - generation stops at the end of the assistant turn (EOS + end-of-turn tokens, stop at `Tool result for`, keep only the first tool call). Run 3 SFT calls all hit the 320-token cap and 16.7% were invalid; an offline replay of Run 3's outputs gives 0% invalid with the other metrics unchanged.
- **B** - each model gets its own name in AuditKIT, so the cache can't serve base results to SFT (the cache is also cleared in step 6).
- **C** - ARC asks for the option number and is scored with AuditKIT `Acc` (Run 3's 0.0 was exact-match on free text).
Because A changes generation, **base and SFT are both re-run** (`REEVAL = True`); earlier results are kept, not deleted.
>
> **After any restart:** run the Config cell, then continue. After a **VM reset**, run steps 0-1 and **R** again.

In [3]:
!nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
import sys; print(sys.version)  # AgentTune needs Python >= 3.12

name, compute_cap, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 8.0, 40960 MiB
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]


## 0 - Get the code
Upload **`safequery_agent.zip`** (~60 KB, the full project with `scripts/`), from
`D:\AI_ML_SME\Lexsi - AI Safety and Alignment Hackathon\`. **Not** `safequery.zip`, which holds only the package.

In [8]:
import os
PROJ = "/content/safequery_agent"
REPO_URL = ""   # optional: a git URL; leave empty to upload the zip

if not os.path.exists(f"{PROJ}/scripts/build_data.py"):
    %cd /content
    if REPO_URL:
        !git clone -q {REPO_URL} {PROJ}
    else:
        from google.colab import files
        uploaded = files.upload()          # choose safequery_agent.zip
        for name in uploaded:
            if name.endswith(".zip"):
                !unzip -q -o "{name}" -d /content

assert os.path.exists(f"{PROJ}/scripts/build_data.py"), (
    "scripts/ is missing - you probably uploaded safequery.zip (package only). "
    "Run: !rm -rf /content/safequery /content/safequery_agent  and upload safequery_agent.zip.")
%cd {PROJ}
!ls . scripts

/content


Saving _bootstrap.py to _bootstrap.py
Saving build_data.py to build_data.py
Saving compare.py to compare.py
Saving eval_benchmarks.py to eval_benchmarks.py
Saving eval_model.py to eval_model.py
Saving merge_adapter.py to merge_adapter.py
Saving push_to_hub.py to push_to_hub.py
Saving roi_report.py to roi_report.py
Saving smoke_test.py to smoke_test.py
Saving train_grpo.py to train_grpo.py
Saving train_sft.py to train_sft.py


AssertionError: scripts/ is missing - you probably uploaded safequery.zip (package only). Run: !rm -rf /content/safequery /content/safequery_agent  and upload safequery_agent.zip.

In [9]:
!rm -rf /content/safequery /content/safequery_agent  and upload safequery_agent.zip.

## 0b - Patch for T4 + transformers 5.x (safety net)
The current zip already includes these fixes, so this changes nothing. Safe to run more than once.

In [7]:
%cd /content/safequery_agent
# Run 5 needs the 2026-11-08 zip with fixes A-C
!grep -c "def trim_turn" safequery/policies.py && grep -c "CallableModel" scripts/eval_benchmarks.py && echo "fixes A-C present" || echo "OLD ZIP - upload the new safequery_agent.zip"
!sed -i 's/torch.cuda.is_bf16_supported()/torch.cuda.is_bf16_supported(including_emulation=False)/g' safequery/policies.py scripts/train_sft.py scripts/train_grpo.py scripts/merge_adapter.py
!sed -i 's/warmup_ratio=0.05/warmup_steps=0.05/' scripts/train_sft.py
!grep -n "is_bf16_supported\|warmup" safequery/policies.py scripts/*.py

[Errno 2] No such file or directory: '/content/safequery_agent'
/content
1
grep: scripts/eval_benchmarks.py: No such file or directory
OLD ZIP - upload the new safequery_agent.zip
sed: can't read scripts/train_sft.py: No such file or directory
sed: can't read scripts/train_grpo.py: No such file or directory
sed: can't read scripts/merge_adapter.py: No such file or directory
sed: can't read scripts/train_sft.py: No such file or directory
safequery/policies.py:125:            dtype = torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False) else torch.float16
grep: scripts/*.py: No such file or directory


## 1 - Install AgentTune and AuditKIT (unmodified, from Lexsi Labs)
5-15 minutes. The red **dependency conflict** warnings are harmless.
**After it installs, use Runtime -> Restart session**, then continue from the next cell.

In [11]:
!git clone -q https://github.com/Lexsi-Labs/AgentTune.git /content/AgentTune 2>/dev/null || echo "AgentTune already cloned"
!git clone -q https://github.com/Lexsi-Labs/AuditKIT.git /content/AuditKIT 2>/dev/null || echo "AuditKIT already cloned"
import subprocess
if subprocess.run(["python", "-c", "import agenttune, auditkit"], capture_output=True).returncode == 0:
    print("AgentTune + AuditKIT already installed - skipping")
else:
    !pip install -q --only-binary=pandas -e /content/AgentTune -e /content/AuditKIT
    print("Installed. Now: Runtime -> Restart session, then run the next cells.")

AgentTune already cloned
AuditKIT already cloned
AgentTune + AuditKIT already installed - skipping


In [12]:
# Lets this kernel import the editable installs even if you didn't restart
import sys
for p in ["/content/AgentTune/src", "/content/AuditKIT/src", "/content/AuditKIT"]:
    if p not in sys.path: sys.path.insert(0, p)

import torch, transformers, trl, peft, agenttune, auditkit
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "| native bf16", torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False))
print("transformers", transformers.__version__, "| trl", trl.__version__, "| peft", peft.__version__)
# A100 / L4: native bf16 True. T4: False (fp16).

torch 2.11.0+cu130 | cuda True | native bf16 True
transformers 5.22.1 | trl 1.7.1 | peft 0.21.2


## 1b - Hugging Face login
Tiny Aya is **gated** (accept the terms at https://huggingface.co/CohereLabs/tiny-aya-global). Use a **new** fine-grained token; never paste tokens into code cells.

In [13]:
from huggingface_hub import notebook_login
notebook_login()

## Config - **re-run this cell after every restart**

In [14]:
import os, glob, shutil, hashlib, json, torch
%cd /content/safequery_agent

MODEL     = "CohereLabs/tiny-aya-global"
LIMIT     = None    # None = all 111 test tasks (Run 3 used 60)
REEVAL    = True    # re-run base + SFT even if results exist (fix A changes generation)
LOAD_4BIT = False   # True only if an eval runs out of GPU memory
RUN_GRPO  = False   # optional; needs an A100/L4 and spare time
HF_USER   = "Siva2022"

LIMIT_ARG = f"--limit {LIMIT}" if LIMIT else ""
Q_ARG     = "--load-4bit" if LOAD_4BIT else ""
DRIVE_DIR = "/content/drive/MyDrive/safequery"
GPU_NAME  = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"

def _mount():
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

def _md5(p):
    return hashlib.md5(open(p, "rb").read()).hexdigest() if os.path.exists(p) else None

def _n_tasks(run):
    p = f"results/{run}/metrics.json"
    return json.load(open(p))["metrics"]["n_tasks"] if os.path.exists(p) else None

def save_to_drive():
    """Copy results/, results_run3_60/, runs/ (minus merged model + checkpoints) and data jsonl to Drive."""
    _mount(); os.makedirs(DRIVE_DIR, exist_ok=True)
    skip = shutil.ignore_patterns("sft_merged", "checkpoint-*")
    for d in ["results", "runs"] + sorted(glob.glob("results_run3_60") + glob.glob("results_prev_*")):
        if os.path.exists(d):
            shutil.copytree(d, f"{DRIVE_DIR}/{d}", dirs_exist_ok=True, ignore=skip)
    for f in glob.glob("data/*.jsonl"):
        shutil.copy(f, DRIVE_DIR)
    print("saved ->", DRIVE_DIR, sorted(os.listdir(DRIVE_DIR)))

n_test = sum(1 for _ in open("data/tasks_test.jsonl")) if os.path.exists("data/tasks_test.jsonl") else None
print("GPU:", GPU_NAME, "| cwd", os.getcwd(), "| test tasks:", n_test or "- (run step 2)")

[Errno 2] No such file or directory: '/content/safequery_agent'
/content
GPU: NVIDIA A100-SXM4-40GB | cwd /content | test tasks: - (run step 2)


## R - Restore saved files from Drive
Lists what is in `MyDrive/safequery`, then copies it back:
- `runs/` -> `runs/` (SFT adapter, training log)
- `results/` (and `results_run3_60/` if present) -> same names here
- `*.jsonl` at the Drive root -> `drive_data/` (kept aside; step 2 checks the rebuilt tasks against them)

This step only reads from Drive; nothing there is changed.

In [15]:
%cd /content/safequery_agent
_mount()
assert os.path.isdir(DRIVE_DIR), f"{DRIVE_DIR} not found - check the Drive account, or that Run 3 saved there"

print("== Drive contents ==")
for root, dirs, files in os.walk(DRIVE_DIR):
    dirs[:] = sorted(d for d in dirs if not d.startswith("checkpoint-"))
    if root[len(DRIVE_DIR):].count(os.sep) > 2:
        continue
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"  {os.path.relpath(p, DRIVE_DIR):60s} {os.path.getsize(p)/1e6:8.2f} MB")

for d in ["runs", "results"] + [os.path.basename(p) for p in glob.glob(f"{DRIVE_DIR}/results_run3_60") + glob.glob(f"{DRIVE_DIR}/results_prev_*")]:
    if os.path.isdir(f"{DRIVE_DIR}/{d}"):
        shutil.copytree(f"{DRIVE_DIR}/{d}", d, dirs_exist_ok=True)
os.makedirs("drive_data", exist_ok=True)
for f in glob.glob(f"{DRIVE_DIR}/*.jsonl"):
    shutil.copy(f, "drive_data/")

print("\n== Restored ==")
for d in ["runs", "results", "results_run3_60", "drive_data"]:
    print(f"{d}:", sorted(os.listdir(d)) if os.path.exists(d) else "-")

[Errno 2] No such file or directory: '/content/safequery_agent'
/content
Mounted at /content/drive
== Drive contents ==
  grpo_train.jsonl                                                 1.06 MB
  sft_train.jsonl                                                  3.69 MB
  tasks_test.jsonl                                                 0.05 MB
  tasks_train.jsonl                                                0.28 MB
  results/COMPARISON.md                                            0.00 MB
  results/ROI.json                                                 0.00 MB
  results/ROI.md                                                   0.00 MB
  results/base/auditkit_result.json                                0.71 MB
  results/base/benchmarks.json                                     0.00 MB
  results/base/episodes.jsonl                                      0.49 MB
  results/base/metrics.json                                        0.00 MB
  results/sft/auditkit_result.json                     

In [16]:
# Check the SFT adapter is complete before using it
%cd /content/safequery_agent
need = ["runs/sft/adapter_config.json", "runs/sft/adapter_model.safetensors"]
for p in need:
    print(("OK      " if os.path.exists(p) else "MISSING ") + p,
          f"({os.path.getsize(p)/1e6:.1f} MB)" if os.path.exists(p) else "")
assert all(os.path.exists(p) for p in need), "SFT adapter incomplete in Drive - retrain with Run 3 step 4"
cfg = json.load(open("runs/sft/adapter_config.json"))
print("base model:", cfg.get("base_model_name_or_path"), "| r", cfg.get("r"), "| targets", cfg.get("target_modules"))
log = "runs/sft/safequery_train_log.json"
if os.path.exists(log):
    print("train log:", json.dumps(json.load(open(log)))[:300])

[Errno 2] No such file or directory: '/content/safequery_agent'
/content
OK      runs/sft/adapter_config.json (0.0 MB)
OK      runs/sft/adapter_model.safetensors (121.0 MB)
base model: CohereLabs/tiny-aya-global | r 16 | targets ['o_proj', 'down_proj', 'up_proj', 'k_proj', 'gate_proj', 'q_proj', 'v_proj']
train log: {"base_model": "CohereLabs/tiny-aya-global", "n_demonstrations": 600, "epochs": 2.0, "lr": 0.0002, "lora_r": 16, "first_loss": 2.075651931762695, "last_loss": 0.013271866738796234, "log_history": [{"loss": 2.075651931762695, "grad_norm": 0.8403205871582031, "learning_rate": 0.0001, "entropy": 1.5205


In [22]:
%cd /content
from google.colab import files
up = files.upload()                      # choose safequery_agent.zip
name = list(up)[0]
!unzip -q -o "{name}" -d /content
!ls /content/safequery_agent/scripts

/content


Saving safequery.zip to safequery (1).zip
ls: cannot access '/content/safequery_agent/scripts': No such file or directory


In [23]:
%cd /content
!rm -rf /content/safequery "/content/safequery (1).zip" /content/safequery.zip
from google.colab import files
up = files.upload()                      # choose safequery_agent.zip (~72 KB)
name = list(up)[0]
assert "safequery_agent" in name, f"wrong file: {name} - pick safequery_agent.zip"
!unzip -q -o "{name}" -d /content
!ls /content/safequery_agent /content/safequery_agent/scripts

/content


Saving safequery_agent.zip to safequery_agent.zip
/content/safequery_agent:
notebooks  pyproject.toml  README.md  safequery  scripts

/content/safequery_agent/scripts:
_bootstrap.py  eval_benchmarks.py  push_to_hub.py  train_grpo.py
build_data.py  eval_model.py	   roi_report.py   train_sft.py
compare.py     merge_adapter.py    smoke_test.py


## 2 - Rebuild the database and check the tasks match Run 3
`retail.db` is not saved to Drive; it is rebuilt from seed 7. The rebuilt task files should be identical to the Drive copies, so before/after numbers are on the same tasks.

In [24]:
%cd /content/safequery_agent
!python scripts/build_data.py --rebuild-db
print()
for f in ["tasks_train.jsonl", "tasks_test.jsonl", "sft_train.jsonl", "grpo_train.jsonl"]:
    a, b = _md5(f"data/{f}"), _md5(f"drive_data/{f}")
    print(f"{f:20s}", "MATCH" if a == b else ("no Drive copy" if b is None else "DIFFERENT - tell Claude before continuing"))
n_test = sum(1 for _ in open("data/tasks_test.jsonl"))
print("test tasks:", n_test)

/content/safequery_agent
I0000 00:00:1794123300.746286    8323 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794123300.815170    8323 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794123303.179455    8323 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[db] data/retail.db  rows: audit_log=50,000, daily_sales=3,655, orders=200,000, products=400, cust

In [25]:
%cd /content/safequery_agent
!python scripts/smoke_test.py

/content/safequery_agent
I0000 00:00:1794123332.659658    8513 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794123332.728787    8513 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794123335.075059    8513 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
[unit] guard: 10 cases OK | authorizer blocks writes OK
[unit] cost: customer lookup=40 rows, mont

## 3 - Keep earlier results, then baseline on all test tasks
Run 3's 60-task results go to `results_run3_60/` (other earlier runs to `results_prev_<n>/`); they are moved to `results_run3_60/` (and saved to Drive), so nothing is lost.

In [26]:
%cd /content/safequery_agent
os.makedirs("results_run3_60", exist_ok=True)
MARK = "results/.run5_archived"   # set once, so re-running this cell never archives Run 5's own results
for run in ["base", "sft"]:
    n = _n_tasks(run)
    if n is not None and ((REEVAL and not os.path.exists(MARK)) or (LIMIT is None and n < n_test)):
        dst = f"results_run3_60/{run}" if n == 60 else f"results_prev_{n}/{run}"
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.exists(dst): shutil.rmtree(dst)
        shutil.move(f"results/{run}", dst)
        print(f"moved results/{run} ({n} tasks) -> {dst}")
for f in ["COMPARISON.md", "ROI.md", "ROI.json"]:
    if os.path.exists(f"results/{f}") and not os.path.exists(f"results_run3_60/{f}"):
        shutil.move(f"results/{f}", f"results_run3_60/{f}")
os.makedirs("results", exist_ok=True); open(MARK, "w").write("archived")
print("results:", sorted(os.listdir("results")), "| kept:", sorted(os.listdir("results_run3_60")))
save_to_drive()

/content/safequery_agent
results: ['.run5_archived', 'smoke_expert', 'smoke_scripted_unsafe'] | kept: []
saved -> /content/drive/MyDrive/safequery ['grpo_train.jsonl', 'results', 'results_run3_60', 'runs', 'sft_train.jsonl', 'tasks_test.jsonl', 'tasks_train.jsonl']


In [27]:
%cd /content/safequery_agent
if _n_tasks("base") and (LIMIT or _n_tasks("base") >= n_test):
    print("baseline already done on", _n_tasks("base"), "tasks - skipping (delete results/base to re-run)")
else:
    !python scripts/eval_model.py --name base --model {MODEL} {LIMIT_ARG} {Q_ARG}
save_to_drive()

/content/safequery_agent
I0000 00:00:1794123357.748129    8670 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794123357.819215    8670 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794123360.162014    8670 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
config.json: 100% 1.96k/1.96k [00:00<00:00, 4.47MB/s]
tokenizer_config.json: 100% 8.17k/8.17k [00:

## 5 - Evaluate the restored SFT adapter on all test tasks

In [31]:
%cd /content/safequery_agent
_mount()
print("Drive:", sorted(os.listdir(DRIVE_DIR)))
print("Drive runs/sft:", os.listdir(f"{DRIVE_DIR}/runs/sft") if os.path.isdir(f"{DRIVE_DIR}/runs/sft") else "NOT IN DRIVE")
print("stray /content/runs/sft:", os.listdir("/content/runs/sft") if os.path.isdir("/content/runs/sft") else "-")

src = f"{DRIVE_DIR}/runs" if os.path.isdir(f"{DRIVE_DIR}/runs/sft") else ("/content/runs" if os.path.isdir("/content/runs/sft") else None)
if src:
    shutil.copytree(src, "runs", dirs_exist_ok=True)
    print("copied from", src, "->", os.listdir("runs/sft"))
else:
    print("No adapter found anywhere")

/content/safequery_agent
Drive: ['grpo_train.jsonl', 'results', 'results_run3_60', 'runs', 'sft_train.jsonl', 'tasks_test.jsonl', 'tasks_train.jsonl']
Drive runs/sft: ['adapter_model.safetensors', 'tokenizer.json', 'adapter_config.json', 'tokenizer_config.json', 'training_args.bin', 'safequery_train_log.json', 'chat_template.jinja', 'README.md']
stray /content/runs/sft: ['training_args.bin', 'tokenizer.json', 'adapter_model.safetensors', 'safequery_train_log.json', 'README.md', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja']
copied from /content/drive/MyDrive/safequery/runs -> ['training_args.bin', 'tokenizer.json', 'adapter_model.safetensors', 'safequery_train_log.json', 'README.md', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja']


In [32]:
%cd /content/safequery_agent
assert os.path.exists("runs/sft/adapter_config.json"), "no SFT adapter - run step R"
if _n_tasks("sft") and (LIMIT or _n_tasks("sft") >= n_test):
    print("SFT eval already done on", _n_tasks("sft"), "tasks - skipping (delete results/sft to re-run)")
else:
    !python scripts/eval_model.py --name sft --model {MODEL} --adapter runs/sft {LIMIT_ARG} {Q_ARG}
save_to_drive()

/content/safequery_agent
I0000 00:00:1794126828.117005   23338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794126828.187020   23338 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794126830.526698   23338 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Loading weights: 100% 290/290 [00:01<00:00, 175.35it/s]
[eval] sft: tiny-aya-global+sft on 111 tas

## 6 - Public benchmarks via AuditKIT: BFCL (tool calling) + ARC (no regression)
AuditKIT caches results on disk by model fingerprint. In Run 3 it served the **base** results to the SFT run (0 model calls).
This cell clears the cache before each model and checks that each run really called the model.

In [33]:
%cd /content/safequery_agent
AK_CACHE = os.path.join(os.environ.get("XDG_CACHE_HOME", os.path.expanduser("~/.cache")), "auditkit", "runs")
for name, extra in [("base", ""), ("sft", "--adapter runs/sft")]:
    shutil.rmtree(AK_CACHE, ignore_errors=True)
    !python scripts/eval_benchmarks.py --name {name} --model {MODEL} {extra} --bfcl-limit 100 --arc-limit 200 {Q_ARG}
    calls = json.load(open(f"results/{name}/benchmarks.json"))["throughput"]["model_calls"]
    b = json.load(open(f"results/{name}/benchmarks.json"))
    print(f"[check] {name}: {calls} model calls", "OK" if calls > 0 else "<- CACHED, numbers not valid",
          "| ARC", b.get("arc_challenge"))
save_to_drive()

/content/safequery_agent
I0000 00:00:1794128889.166928   32160 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794128889.236532   32160 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794128891.609743   32160 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
Loading weights: 100% 290/290 [00:01<00:00, 174.87it/s]
Reconstructing (incomplete total...): |   

## 7 - *Optional*: agentic GRPO with AgentTune
Off by default (`RUN_GRPO = False`). On an A100/L4 these settings could be raised - agree them first.

In [34]:
RUN_GRPO = True
%cd /content/safequery_agent
if RUN_GRPO:
    if not os.path.exists("runs/sft_merged/config.json"):
        !python scripts/merge_adapter.py --model {MODEL} --adapter runs/sft --out runs/sft_merged
    !python scripts/train_grpo.py --model runs/sft_merged --out runs/grpo --max-steps 60 --num-generations 2 --batch-size 2 --max-completion-length 192
    !python scripts/eval_model.py --name grpo --model runs/sft_merged --adapter runs/grpo {LIMIT_ARG} {Q_ARG}
    save_to_drive()
else:
    print("RUN_GRPO is False - skipping")

/content/safequery_agent
Loading weights: 100% 290/290 [00:01<00:00, 170.20it/s]
Writing model shards: 100% 1/1 [00:22<00:00, 22.84s/it]
[merge] runs/sft + CohereLabs/tiny-aya-global -> runs/sft_merged
I0000 00:00:1794138539.791939   72382 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794138539.862221   72382 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794138542.256329   72382 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation ord

## 8 - Before / after table and ROI
Prices are **placeholders** - use current published prices (with the date) before quoting the ROI. `GPU_NAME` comes from this runtime; set `GPU_PRICE` to its hourly price.

In [35]:
%cd /content/safequery_agent
RUN_GRPO = True
GPU_PRICE = 0.35   # placeholder $/h - update for the GPU you ran on
RUNS = "base sft" + (" grpo" if RUN_GRPO else "")
!python scripts/compare.py --runs {RUNS} --out results/COMPARISON.md
!python scripts/roi_report.py --run results/sft --closed-in 3.0 --closed-out 15.0 --gpu-name "{GPU_NAME}" --gpu-price {GPU_PRICE} --out results/ROI.md
from IPython.display import Markdown, display
display(Markdown(open("results/COMPARISON.md", encoding="utf-8").read()))
display(Markdown(open("results/ROI.md", encoding="utf-8").read()))
save_to_drive()

/content/safequery_agent
I0000 00:00:1794142101.056861   87781 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1794142101.125547   87781 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1794142103.494989   87781 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
## SafeQuery benchmark (held-out test set)

| Metric | base | sft | grpo |
|---|---|---|---|
| Tas

## SafeQuery benchmark (held-out test set)

| Metric | base | sft | grpo |
|---|---|---|---|
| Task success ↑ | 19.8% | 99.1% | 99.1% |
| Answer accuracy (verified) ↑ | 10.5% | 100.0% | 100.0% |
| Correct refusals ↑ | 64.0% | 96.0% | 96.0% |
| Injection resistance ↑ | 0.0% | 100.0% | 100.0% |
| Unsafe SQL attempts ↓ | 0.9% | 0.9% | 0.9% |
| Over-refusal ↓ | 0.0% | 0.0% | 0.0% |
| Over-cost-threshold queries ↓ | 1.2% | 0.0% | 0.0% |
| Full scans executed ↓ | 0.0% | 0.0% | 0.0% |
| Intent accuracy ↑ | 49.5% | 99.1% | 99.1% |
| Invalid tool calls ↓ | 0.0% | 0.0% | 0.0% |
| Avg tool calls / task ↓ | 1.06 | 1.00 | 1.00 |
| auditkit_agent_loop_detection | 1.000 | 1.000 | 1.000 |
| auditkit_completed_rows | 111 | 111 | 111 |
| auditkit_outcome_success | 0.198 | 0.991 | 0.991 |
| auditkit_redundant_tool_calls | 0.000 | 0.000 | 0.000 |
| auditkit_tool_call_exact_name | 0.171 | 0.991 | 0.991 |
| auditkit_tool_call_f1_name | 0.213 | 0.991 | 0.991 |
| auditkit_tool_call_validity | 1.000 | 1.000 | 1.000 |

## Public benchmarks (AuditKIT)

| Benchmark | base | sft | grpo |
|---|---|---|---|
| arc_challenge · acc | 0.156 | 0.126 | - |
| bfcl_irrelevance · tool_call_exact_custom_794bf7e6 | 0.160 | 0.220 | - |
| bfcl_irrelevance · tool_call_f1_custom_794bf7e6 | 0.160 | 0.220 | - |
| bfcl_irrelevance · tool_call_precision_custom_794bf7e6 | 0.160 | 0.220 | - |
| bfcl_irrelevance · tool_call_recall_custom_794bf7e6 | 0.160 | 0.220 | - |
| bfcl_simple · tool_call_exact_custom_794bf7e6 | 0.920 | 0.450 | - |
| bfcl_simple · tool_call_f1_custom_794bf7e6 | 0.920 | 0.450 | - |
| bfcl_simple · tool_call_precision_custom_794bf7e6 | 0.920 | 0.450 | - |
| bfcl_simple · tool_call_recall_custom_794bf7e6 | 0.920 | 0.450 | - |


## ROI: closed model vs self-hosted fine-tuned Tiny Aya

| | Closed model (token billing) | Fine-tuned Tiny Aya (self-hosted) |
|---|---|---|
| Monthly volume | 100,000 questions, 200,000 LLM calls | same |
| Tokens / month | 332,600,000 in + 8,000,000 out | same |
| Pricing used | $3.0/M in, $15.0/M out | NVIDIA A100-SXM4-40GB at $0.35/h, 6.65 tok/s x 8 streams |
| GPU busy hours / month | - | 128.6 h |
| **Monthly cost** | **$1,117.8** | **$255.5** always-on ($45.01 if billed per busy hour) |
| Cost per 1k questions | $11.178 | $2.555 |
| One-off training | - | $1.05 |

Assumptions: prices are inputs (see flags); throughput and tokens per call are measured from the eval run.


saved -> /content/drive/MyDrive/safequery ['grpo_train.jsonl', 'results', 'results_run3_60', 'runs', 'sft_train.jsonl', 'tasks_test.jsonl', 'tasks_train.jsonl']


## 9 - Download results for analysis
`safequery_results_run5.zip` holds the new results, the kept Run 3 results and the task files.

In [36]:
%cd /content/safequery_agent
!zip -r -q /content/safequery_results_run5.zip results results_run3_60 results_prev_* data/*.jsonl
!ls -lh /content/safequery_results_run5.zip
from google.colab import files
files.download("/content/safequery_results_run5.zip")

/content/safequery_agent
-rw-r--r-- 1 root root 469K Nov  8 12:49 /content/safequery_results_run5.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10 - Publish to Hugging Face - **only after the numbers are final and you have decided what to share**
`push_to_hub.py` uploads the adapter + model card, the task/SFT dataset (incl. the database build script) and the full eval traces. Run 3 already created these repos as **public**. Re-running uploads new files but does not change a repo's visibility.

In [37]:
%cd /content/safequery_agent
PUSH = True   # set True when the results are final
RUN_GRPO = True
ADAPTERS = "sft=runs/sft" + (" grpo=runs/grpo" if RUN_GRPO else "")
if PUSH and HF_USER:
    !python scripts/push_to_hub.py --user {HF_USER} --adapters {ADAPTERS}
else:
    print("PUSH is False - skipping")

/content/safequery_agent
Found 8 files to upload
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ████████████████████  -
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 3 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ░░░░░░░░░░░░░░░░░░░░  0 / 3 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ██████░░░░░░░░░░░░░░  1 / 3 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   █████████████░░░░░░░  2 / 3 files
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ████████████████████  3 / 3 files ✓
  Validating  ░░░░░░░░░░░░░░░░░░░░  0%
  Committing  ░░░░░░░░░░░░░░░░░░░░  0 / 8
  Preparing   ████████████████████  8 / 8 ✓
  Uploading   ████████████████████  3 / 3 files ✓
  Validating  ████████